In [1]:
## Imports 

from fastapi import FastAPI
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from fastapi.responses import StreamingResponse

In [2]:
## Fast API initialize

app = FastAPI()

In [3]:
## Initialize the LLM

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [4]:
## create the prompt 

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])

In [5]:
## LLM chain 

chain = prompt | llm | StrOutputParser()

In [6]:
## Request model
class AskReq(BaseModel):
    question: str

In [7]:
## Create /ask
@app.post("/ask")
async def ask(req: AskReq):

    answer = await chain.ainvoke(
        {
            "question": req.question
        }
    )

    return {
        "answer": answer
    }

In [8]:
## streaming endpoint 

async def gen(question):

    async for chunk in chain.astream(
        {
            "question": question
        }
    ):

        yield f"data: {chunk}\n\n"

In [9]:
@app.post("/stream")
async def stream(req: AskReq):

    return StreamingResponse(
        gen(req.question),
        media_type="text/event-stream"
    )

In [10]:
## health endpont 
@app.get("/health")
def health():

    return {
        "status": "ok",
        "model": "llama3.2"
    }

In [11]:
import uvicorn
import nest_asyncio

# This allows uvicorn to run inside the existing Jupyter event loop
nest_asyncio.apply()

uvicorn.run(app, host="127.0.0.1", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [1]:

import nest_asyncio
nest_asyncio.apply()

# Cell 2: Define your FastAPI app and routes
from fastapi import FastAPI
app = FastAPI()

@app.get("/")
async def root():
    return {"message": "Hello World"}

# Cell 3: Run the server
import uvicorn
# Use the config dictionary approach for better stability in notebooks
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [36716]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64123 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64123 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36716]


In [1]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import StreamingResponse
import uvicorn

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

app = FastAPI()
llm = ChatOllama(model="llama3.2", temperature=0)
chain = ChatPromptTemplate.from_template("{question}") | llm | StrOutputParser()

class AskReq(BaseModel):
    question: str

In [2]:
@app.post("/ask")
async def ask(req: AskReq):
    answer = await chain.ainvoke({"question": req.question})
    return {"answer": answer}

@app.post("/stream")
async def stream(req: AskReq):
    async def gen():
        async for chunk in chain.astream({"question": req.question}):
            yield f"data: {chunk}\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

@app.get("/health")
def health():
    return {"status": "ok", "model": "llama3.2"}

In [3]:
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [32976]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51256 - "GET / HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [32976]


In [1]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import StreamingResponse
import uvicorn

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# FastAPI app
app = FastAPI()

# LLM
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# Chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()

# Request model
class AskReq(BaseModel):
    question: str

# Ask endpoint
@app.post("/ask")
async def ask(req: AskReq):
    answer = await chain.ainvoke({"question": req.question})
    return {"answer": answer}

# Stream endpoint
@app.post("/stream")
async def stream(req: AskReq):

    async def gen():
        async for chunk in chain.astream({"question": req.question}):
            yield f"data: {chunk}\n\n"

    return StreamingResponse(
        gen(),
        media_type="text/event-stream"
    )

# Health endpoint
@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "llama3.2"
    }

# Start server
config = uvicorn.Config(
    app=app,
    host="127.0.0.1",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()



INFO:     Started server process [35396]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64687 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:64687 - "GET /ask HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:61644 - "GET /stream HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:64819 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:65387 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:65387 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:53562 - "GET /stream HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:59787 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:59787 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:61766 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:64442 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:60981 - "POST /stream HTTP/1.1" 200 OK
INFO:     127.0.0.1:54524 - "GET /stream HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:54524 - "GET /stream HTTP/1.1" 405 Method Not Allowed


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [35396]
